# Fairness Evaluation on UCI Adult Dataset
This notebook:
- Loads and preprocesses the UCI Adult dataset
- Trains a classifier to predict income level
- Evaluates fairness using Fairlearn

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference, equalized_odds_difference

## 1. Load and Preprocess the Data

In [3]:
# Load dataset
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]

df = pd.read_csv("adult/adult.data", header=None, names=column_names, na_values=" ?")
df = df.dropna().reset_index(drop=True)
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

## 2. Encode and Split Data

In [4]:
# Label encoding for categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.drop("income")
df_encoded = df.copy()
le = LabelEncoder()
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Target variable
df_encoded["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# Features and labels
X = df_encoded.drop(columns="income")
y = df_encoded["income"]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
sensitive = df.loc[X_test.index, "education"]

## 3. Train Classifier and Predict

In [6]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 4. Fairness Evaluation

In [7]:
mf = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive
)

print("Accuracy by group:")
print(mf.by_group["accuracy"])

print("\nSelection rate by group:")
print(mf.by_group["selection_rate"])

print("\nDemographic parity difference:",
      demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive))

print("Equalized odds difference:",
      equalized_odds_difference(y_test, y_pred, sensitive_features=sensitive))

Accuracy by group:
education
10th            0.945833
11th            0.968354
12th            0.933333
1st-4th         0.956522
5th-6th         0.989796
7th-8th         0.929936
9th             0.968504
Assoc-acdm      0.825279
Assoc-voc       0.802005
Bachelors       0.810900
Doctorate       0.754545
HS-grad         0.859416
Masters         0.838776
Preschool       1.000000
Prof-school     0.834395
Some-college    0.851815
Name: accuracy, dtype: float64

Selection rate by group:
education
10th            0.016667
11th            0.015823
12th            0.028571
1st-4th         0.000000
5th-6th         0.020408
7th-8th         0.012739
9th             0.007874
Assoc-acdm      0.245353
Assoc-voc       0.187970
Bachelors       0.418253
Doctorate       0.809091
HS-grad         0.111406
Masters         0.616327
Preschool       0.000000
Prof-school     0.802548
Some-college    0.143145
Name: selection_rate, dtype: float64

Demographic parity difference: 0.8090909090909091
Equalized odds d